In [ ]:
import numpy as np
from PIL import Image

def frequency_sharpening(image_path, filter_type="butterworth", D0=30, n=2):
    """Применяет частотное повышение резкости к изображению."""
    
    # 1. Загрузка изображения и преобразование в grayscale
    img = Image.open(image_path).convert("L")
    f = np.array(img, dtype=np.float32)
    
    # 2. Предварительная обработка: умножение на (-1)^(x+y)
    M, N = f.shape
    x = np.arange(M)[:, np.newaxis]  
    y = np.arange(N)                 
    f_preprocessed = f * (-1)**(x + y)
    
    # 3. Вычисление ДПФ
    F = np.fft.fft2(f_preprocessed)
    
    # 4. Создание фильтра частот
    u = np.arange(M)[:, np.newaxis]
    v = np.arange(N)
    D = np.sqrt((u - M//2)**2 + (v - N//2)**2)
    
    if filter_type == "ideal":
        H = (D > D0).astype(np.float32)
    elif filter_type == "butterworth":
        H = 1 / (1 + (D0 / (D + 1e-6))**(2*n))
    elif filter_type == "gaussian":
        H = 1 - np.exp(-D**2 / (2 * D0**2))
    else:
        raise ValueError("Неизвестный тип фильтра")
    
    # 5. Применение фильтра
    G = F * H
    
    # 6. Обратное ДПФ
    g_preprocessed = np.fft.ifft2(G).real
    
    # 7. Заключительная обработка: умножение на (-1)^(x+y)
    g = g_preprocessed * (-1)**(x + y)
    
    # 8. Нормализация и преобразование к uint8
    g_normalized = np.clip(g, 0, 255)
    
    return g_normalized.astype(np.uint8)

In [ ]:
# Параметры
name = "novak"
filter_type = "gaussian"

input_path = f"origins/{name}.jpg"
output_path = f"results/{name}_{filter_type}_output.jpg"

# Применение
sharpened_image = frequency_sharpening(input_path, filter_type=filter_type, D0=30, n=2)

# Сохранение результата
Image.fromarray(sharpened_image).save(output_path)